# Exploratory Data Analysis: Rideshare Fares and Weather

This notebook explores how Chicago's weather relates to rideshare fares in 2025. It combines two cleaned data sets:

| Data set | File | Grain |
|---|---|---|
| Rideshare trips | rideshare_2025_clean.parquet | One row per trip (over 55 million rows) |
| Weather flags | chicago_weather_2025_daily.csv | One row per day (365 rows) |

The questions I want to answer:

1. Do rain, snow, and cold change the average fare?
2. Is the fare change a pricing change, or a change in the kind of trips people take (distance and duration)?
3. How do fares, and the effect of the weather, change over the seasons?

## Import libraries

In [62]:
# Import the libraries for querying, wrangling, and plotting
import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
from pathlib import Path

### Set folders

In [63]:
PROCESSED_DIR = Path('../data/processed')

# Load the data

## Connect DuckDB to the rideshare data

The rideshare data set is too large to load into a pandas data frame. Instead, DuckDB reads the parquet file directly, and I will do the heavy lifting in SQL. Only small summary tables will be brought into pandas.

In [64]:
con = duckdb.connect()

con.execute(f"""
    CREATE OR REPLACE VIEW rideshare AS
    SELECT *
    FROM read_parquet('{PROCESSED_DIR}/rideshare_2025_clean.parquet')
""")

##### How many rows are in the rideshare data?

In [66]:
con.sql('SELECT COUNT(*) AS trips FROM rideshare')

tripsBIGINT
55075632


## Load the processed weather data

The weather file is tiny, so it goes straight into pandas. It has one row per day with four true/false flags.

In [67]:
weather = pd.read_csv(
    PROCESSED_DIR / 'chicago_weather_2025_daily.csv',
    encoding = 'utf-8-sig',
    parse_dates = ['date']
)

In [68]:
weather.head()

,date,light_rain,heavy_rain,snow,below_freezing
0,2025-01-01,True,False,False,True
1,2025-01-02,False,False,False,True
2,2025-01-03,False,False,True,True
3,2025-01-04,False,False,False,True
4,2025-01-05,False,False,False,True


In [69]:
weather.info()

<class 'pandas.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   date            365 non-null    datetime64[us]
 1   light_rain      365 non-null    bool          
 2   heavy_rain      365 non-null    bool          
 3   snow            365 non-null    bool          
 4   below_freezing  365 non-null    bool          
dtypes: bool(4), datetime64[us](1)
memory usage: 4.4 KB


##### How many days are flagged for each condition?

In [70]:
weather.drop(columns = 'date').sum()

light_rain        105
heavy_rain         16
snow               27
below_freezing    105
dtype: int64

# Aggregate the rideshare data

## Aggregate by date and hour

Every question in this notebook compares days, hours, or seasons, so I scan the 55 million rows once and keep one row for each combination of date and hour. That is at most 365 days x 24 hours, or 8,760 rows, which is tiny for pandas.

I keep totals instead of averages. Totals can be added up to any coarser level (day, month, or season), and averages can be calculated afterward so that each trip is weighted equally.

| Column | Meaning |
|---|---|
| trip_count | Number of trips |
| fare_total | Sum of fares in dollars |
| miles_total | Sum of trip miles |
| seconds_total | Sum of trip seconds |

I also keep only trips that started in 2025.

In [71]:
trips_by_slice = con.sql("""
    SELECT
        CAST(trip_start_timestamp AS DATE) AS date,
        EXTRACT(HOUR FROM trip_start_timestamp) AS hour,
        COUNT(*) AS trip_count,
        CAST(SUM(fare) AS DOUBLE) AS fare_total,
        CAST(SUM(trip_miles) AS DOUBLE) AS miles_total,
        CAST(SUM(trip_seconds) AS DOUBLE) AS seconds_total
    FROM rideshare
    WHERE trip_start_timestamp >= '2025-01-01'
      AND trip_start_timestamp < '2026-01-01'
    GROUP BY ALL
""").df()

In [72]:
# make sure the dates are datetimes so they match the weather data
trips_by_slice['date'] = pd.to_datetime(trips_by_slice['date'])

In [73]:
trips_by_slice.head()

,date,hour,trip_count,fare_total,miles_total,seconds_total
0,2025-01-25,17,13631,172670.0,58919.0,13115058.0
1,2025-01-25,14,9185,121327.5,46286.7,9096673.0
2,2025-01-20,17,9017,129692.5,50262.2,8701029.0
3,2025-01-20,14,7162,104847.5,43411.5,6860568.0
4,2025-01-16,12,6431,96055.0,40988.2,6012387.0


In [74]:
trips_by_slice.info()

<class 'pandas.DataFrame'>
RangeIndex: 8759 entries, 0 to 8758
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           8759 non-null   datetime64[us]
 1   hour           8759 non-null   int64         
 2   trip_count     8759 non-null   int64         
 3   fare_total     8759 non-null   float64       
 4   miles_total    8759 non-null   float64       
 5   seconds_total  8759 non-null   float64       
dtypes: datetime64[us](1), float64(3), int64(2)
memory usage: 410.7 KB


##### Did the aggregation keep all of the trips?

The trips in the summary table should match the rows in the full data. A difference would mean some trips started outside 2025.

In [75]:
total_rows = con.sql('SELECT COUNT(*) FROM rideshare').fetchall()[0][0]
aggregated_trips = trips_by_slice['trip_count'].sum()

print(f'Rows in the rideshare data: {total_rows:,}')
print(f'Trips in the summary table: {aggregated_trips:,}')

Rows in the rideshare data: 55,075,632
Trips in the summary table: 55,075,632


## Median fare per day

The average fare can be pulled up by a small number of very long trips, such as an airport run. A median is not affected by those trips, so I calculate it as a check on the average.

A median cannot be rebuilt from totals, so it needs its own query that scans the data again.

In [76]:
daily_median = con.sql("""
    SELECT
        CAST(trip_start_timestamp AS DATE) AS date,
        CAST(MEDIAN(fare) AS DOUBLE) AS median_fare
    FROM rideshare
    WHERE trip_start_timestamp >= '2025-01-01'
      AND trip_start_timestamp < '2026-01-01'
    GROUP BY ALL
""").df()

daily_median['date'] = pd.to_datetime(daily_median['date'])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [77]:
daily_median.head()

,date,median_fare
0,2025-01-01,15.0
1,2025-01-02,10.0
2,2025-01-03,10.0
3,2025-01-04,10.0
4,2025-01-05,10.0


# Combine rideshare and weather

## Build the daily table

Weather is recorded once per day, so the daily table is the main one for comparing weather conditions. I add up the totals for each day, then divide to get the averages. Dividing totals by the trip count makes these trip-weighted averages.

| Column | Calculation | Meaning |
|---|---|---|
| avg_fare | fare total / trips | Average fare per trip |
| median_fare | from the query above | Fare of the middle trip |
| fare_per_mile | fare total / miles total | Dollars paid per mile |
| avg_miles | miles total / trips | Average trip distance |
| avg_minutes | seconds total / 60 / trips | Average trip duration |

In [78]:
daily = (
    trips_by_slice
    .groupby('date')[['trip_count', 'fare_total', 'miles_total', 'seconds_total']]
    .sum()
    .reset_index()
)

daily['avg_fare'] = daily['fare_total'] / daily['trip_count']
daily['fare_per_mile'] = daily['fare_total'] / daily['miles_total']
daily['avg_miles'] = daily['miles_total'] / daily['trip_count']
daily['avg_minutes'] = daily['seconds_total'] / 60 / daily['trip_count']

daily = daily.merge(daily_median, on = 'date', how = 'left')

## Join the weather flags

I use a left join so every rideshare day is kept, and then check that every day found a weather match.

In [79]:
daily = daily.merge(weather, on = 'date', how = 'left')

In [80]:
print(daily.shape)
daily[['light_rain', 'heavy_rain', 'snow', 'below_freezing', 'median_fare']].isna().sum()

(365, 14)


light_rain        0
heavy_rain        0
snow              0
below_freezing    0
median_fare       0
dtype: int64

There should be 365 rows and no missing values. If any weather flags are missing, the weather file is missing those dates.

## Add calendar information

Fares differ by weekday, and they change over the year, so am I looking at both for comparisons. For ease/readability, I am adding the weekday, the month, and the season.

| Season | Months |
|---|---|
| Winter | December, January, February |
| Spring | March, April, May |
| Summer | June, July, August |
| Fall | September, October, November |

The data only covers 2025, so winter is January and February plus December, which are not back to back.

In [81]:
SEASON_ORDER = ['Winter', 'Spring', 'Summer', 'Fall']

season_by_month = {
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Fall', 10: 'Fall', 11: 'Fall'
}

month_names = {
    1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
    7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
}

In [82]:
daily['weekday'] = daily['date'].dt.day_name()
daily['day_type'] = np.where(daily['date'].dt.dayofweek >= 5, 'Weekend', 'Weekday')
daily['month'] = daily['date'].dt.month

daily['season'] = pd.Categorical(
    daily['month'].map(season_by_month),
    categories = SEASON_ORDER,
    ordered = True
)

## Create one weather condition per day

It is easier to compare four groups than four separate flags. In the cleaning notebook, rain flags were turned off on snow days and light and heavy rain cannot both be True, so every day fits exactly one of these conditions:

| Condition | Rule |
|---|---|
| Snow | snow is True |
| Heavy rain | heavy_rain is True |
| Light rain | light_rain is True |
| Dry | None of the above |

below_freezing is kept as its own column, since a day can be both cold and dry, or cold and snowy. That lets me look at the effect of cold separately from the effect of rain or snow.

In [ ]:
CONDITION_ORDER = ['Dry', 'Light rain', 'Heavy rain', 'Snow']

CONDITION_COLORS = {
    'Dry': '#9e9e9e',
    'Light rain': '#4c9be8',
    'Heavy rain': '#1f3b8c',
    'Snow': '#2bb3a3'
}

In [84]:
daily['condition'] = np.select(
    [daily['snow'], daily['heavy_rain'], daily['light_rain']],
    ['Snow', 'Heavy rain', 'Light rain'],
    default = 'Dry'
)

daily['condition'] = pd.Categorical(
    daily['condition'],
    categories = CONDITION_ORDER,
    ordered = True
)

daily['weather_day'] = daily['condition'] != 'Dry'

##### How many days fall into each condition?

In [85]:
daily['condition'].value_counts().sort_index()

condition
Dry           217
Light rain    105
Heavy rain     16
Snow           27
Name: count, dtype: int64

##### How are the weather days spread across the seasons?

In [86]:
pd.crosstab(daily['season'], daily['condition'], margins = True)

condition,Dry,Light rain,Heavy rain,Snow,All
season,,,,,
Winter,55,15,1,19,90
Spring,54,31,4,3,92
Summer,48,35,9,0,92
Fall,60,24,2,5,91
All,217,105,16,27,365


##### How many days are below freezing in each season?

In [87]:
pd.crosstab(daily['season'], daily['below_freezing'], margins = True)

below_freezing,False,True,All
season,,,
Winter,10,80,90
Spring,76,16,92
Summer,92,0,92
Fall,82,9,91
All,260,105,365


Weather days are not spread evenly. Snow and freezing days only happen in the colder seasons, and some season and condition combinations have only a handful of days. Also, unusual days (a holiday, a big event) are not spread evently accross seasons.

# Fares across the year

## Daily average fare

I start by plotting the average fare for every day of the year and marking the days with rain or snow. The 7-day average smooths out the weekday and weekend swings so the longer trend is easier to see.

In [88]:
daily['avg_fare_7d'] = daily['avg_fare'].rolling(7, center = True, min_periods = 1).mean()

fig = px.line(
    daily,
    x = 'date',
    y = 'avg_fare',
    title = 'Daily Average Rideshare Fare in Chicago, 2025',
    labels = {
        'date': 'Date',
        'avg_fare': 'Average Fare ($)'
    }
)

fig.update_traces(name = 'Daily average', showlegend = True)

fig.add_scatter(
    x = daily['date'],
    y = daily['avg_fare_7d'],
    mode = 'lines',
    name = '7-day average',
    line = dict(width = 3, color = 'black')
)

# add a marker on each day with rain or snow
weather_days = px.scatter(
    daily[daily['weather_day']],
    x = 'date',
    y = 'avg_fare',
    color = 'condition',
    color_discrete_map = CONDITION_COLORS,
    category_orders = {'condition': CONDITION_ORDER}
)

for trace in weather_days.data:
    fig.add_trace(trace)

fig.update_layout(
    title = dict(x = 0.5, xanchor = 'center'),
    legend_title_text = 'Weather',
    height = 500
)

fig.show()

Takeaway: Average fare varies a lot from day to day, from \$12.46 on January 18 to \$24.41 on May 16. The 7-day average shows a clear seasonal pattern. Fares are lowest in January (about \$14.28 at the low point) and climb through the spring to a peak in mid-May. Rain and snow days are scattered across the whole range instead of clustering at the top or bottom. The daily and seasonal swings are much larger than anything the daily weather seems to do.

## Make a fair comparison

Comparing the average fare on snow days to the average on all other days would be misleading, because the two groups are not otherwise alike:

- Weekday mix: fares are different on weekdays and weekends, so a group with more weekend days would look different for that reason alone.
- Temperature and season: snow only happens when it is cold, and cold days are also winter days. Fares in winter may differ from fares in summer regardless of the weather.

I compare each day to the average of dry days in a matching group. There is more than one reasonable group, and each one answers a different question, so I use all of them:

| Comparison | Weather day is compared to | Question it answers |
|---|---|---|
| all | All dry days with the same weekday | How different is this day from a typical dry day, including the cold that comes with the weather? |
| temp | Dry days with the same weekday and the same freezing status | What do rain or snow add once the temperature is held fixed? |
| warm | Dry days above freezing with the same weekday | What does cold alone change? (used for dry days below freezing) |


Comparing against all dry days shows the overall change people experience when snow arrives. Comparing against dry days at the same temperature isolates the snow itself. The gap between the two is the part that the cold explains.

Differences are shown as a percent of the baseline. This does not account for holidays or events, but it is a much fairer starting point than raw averages.

In [89]:
metrics = [
    'avg_fare', 'median_fare', 'fare_per_mile',
    'avg_miles', 'avg_minutes', 'trip_count'
]

dry = daily[daily['condition'] == 'Dry']
dry_warm = dry[~dry['below_freezing']]


def make_baseline(data, keys, name):
    # average of the dry days in each group
    return (
        data
        .groupby(keys, observed = True)[metrics]
        .mean()
        .add_suffix(f'_{name}')
        .reset_index()
    )

In [90]:
baseline_keys = {
    'all': ['weekday'],
    'temp': ['weekday', 'below_freezing'],
    'warm': ['weekday'],
    'season': ['season', 'weekday']
}

baselines = {
    'all': make_baseline(dry, baseline_keys['all'], 'all'),
    'temp': make_baseline(dry, baseline_keys['temp'], 'temp'),
    'warm': make_baseline(dry_warm, baseline_keys['warm'], 'warm'),
    'season': make_baseline(dry, baseline_keys['season'], 'season')
}

##### What do the baselines look like?

Here is the average dry-day fare for each weekday, with and without freezing temperatures.

In [91]:
baselines['temp'][['weekday', 'below_freezing', 'avg_fare_temp', 'fare_per_mile_temp']].round(2)

,weekday,below_freezing,avg_fare_temp,fare_per_mile_temp
0,Friday,False,17.72,2.96
1,Friday,True,16.91,3.05
2,Monday,False,19.08,2.61
3,Monday,True,16.81,2.53
4,Saturday,False,15.99,3.12
5,Saturday,True,14.57,3.09
6,Sunday,False,17.84,2.76
7,Sunday,True,15.68,2.66
8,Thursday,False,18.82,2.83
9,Thursday,True,16.10,2.67


##### Attach the baselines and calculate the differences

For every day and every baseline, I calculate the percent difference for each measure. For the average fare I also keep the difference in dollars.

In [92]:
for name, table in baselines.items():
    daily = daily.merge(table, on = baseline_keys[name], how = 'left')

    # percent difference from the matching dry-day baseline
    for metric in metrics:
        daily[f'{metric}_vs_{name}'] = (daily[metric] / daily[f'{metric}_{name}'] - 1) * 100

    # difference in dollars for the average fare
    daily[f'avg_fare_diff_{name}'] = daily['avg_fare'] - daily[f'avg_fare_{name}']

##### Are there days without a matching dry baseline?

A day has no baseline if there were no dry days in its matching group. Those days would be left out of the comparisons.

In [93]:
baseline_columns = [f'avg_fare_{name}' for name in baselines]

daily.loc[
    daily[baseline_columns].isna().any(axis = 1),
    ['date', 'weekday', 'season', 'condition', 'below_freezing']
]

,date,weekday,season,condition,below_freezing


# How do rain, snow, and cold change fares?

## Summary of fare changes

Each row is a group of days compared to a baseline. A value of -10 means 10% lower than a similar dry day.

- Cold dry day: dry days below freezing, compared to dry days above freezing. This is the effect of cold on its own.
- Light rain, heavy rain, snow: compared two ways, to all dry days and to dry days at the same temperature. Every snow day was below freezing, so for snow the second comparison is to dry days below freezing.

I also include the median fare, the fare per mile, the trip distance, the trip duration, and the number of trips, so the fare change can be put in context.

In [94]:
scenarios = [
    # (weather, compared to, which days, baseline)
    ('Cold dry day', 'dry days above freezing', (daily['condition'] == 'Dry') & daily['below_freezing'], 'warm'),
    ('Light rain', 'all dry days', daily['condition'] == 'Light rain', 'all'),
    ('Light rain', 'dry days, same temperature', daily['condition'] == 'Light rain', 'temp'),
    ('Heavy rain', 'all dry days', daily['condition'] == 'Heavy rain', 'all'),
    ('Heavy rain', 'dry days, same temperature', daily['condition'] == 'Heavy rain', 'temp'),
    ('Snow', 'all dry days', daily['condition'] == 'Snow', 'all'),
    ('Snow', 'dry days, same temperature', daily['condition'] == 'Snow', 'temp')
]

rows = []

for weather_label, compared_to, mask, name in scenarios:
    subset = daily[mask]

    rows.append({
        'weather': weather_label,
        'compared_to': compared_to,
        'days': len(subset),
        'avg_fare_diff': subset[f'avg_fare_diff_{name}'].mean(),
        'avg_fare_pct': subset[f'avg_fare_vs_{name}'].mean(),
        'median_fare_pct': subset[f'median_fare_vs_{name}'].mean(),
        'fare_per_mile_pct': subset[f'fare_per_mile_vs_{name}'].mean(),
        'avg_miles_pct': subset[f'avg_miles_vs_{name}'].mean(),
        'avg_minutes_pct': subset[f'avg_minutes_vs_{name}'].mean(),
        'trip_count_pct': subset[f'trip_count_vs_{name}'].mean()
    })

fare_effects = pd.DataFrame(rows).round(2)

fare_effects

,weather,compared_to,days,avg_fare_diff,avg_fare_pct,median_fare_pct,fare_per_mile_pct,avg_miles_pct,avg_minutes_pct,trip_count_pct
0,Cold dry day,dry days above freezing,63,-1.82,-10.04,-6.08,-1.41,-8.72,-13.07,-1.57
1,Light rain,all dry days,105,0.36,2.07,2.53,2.86,-0.66,0.72,2.94
2,Light rain,"dry days, same temperature",105,0.11,0.67,1.76,2.77,-1.99,-1.01,3.43
3,Heavy rain,all dry days,16,0.84,4.95,1.42,4.35,0.69,2.86,8.52
4,Heavy rain,"dry days, same temperature",16,0.39,2.32,-0.28,3.89,-1.35,-0.53,7.78
5,Snow,all dry days,27,-0.68,-3.87,-3.46,2.75,-6.61,-8.33,1.02
6,Snow,"dry days, same temperature",27,0.84,5.36,3.29,4.91,0.09,3.45,5.24


### Average fare compared to a dry day

The bars show the average percent difference in the average fare. Each weather type has two bars, one for each comparison, so the overall change and the change at the same temperature can be seen side by side.

In [95]:
fig = px.bar(
    fare_effects,
    x = 'weather',
    y = 'avg_fare_pct',
    color = 'compared_to',
    barmode = 'group',
    text = 'avg_fare_pct',
    category_orders = {'weather': ['Cold dry day', 'Light rain', 'Heavy rain', 'Snow']},
    hover_data = {
        'days': True,
        'avg_fare_diff': ':.2f'
    },
    title = 'Change in Average Fare Compared to Dry Days',
    labels = {
        'weather': 'Weather',
        'avg_fare_pct': 'Average Difference in Fare (%)',
        'compared_to': 'Compared to',
        'days': 'Days',
        'avg_fare_diff': 'Difference ($)'
    }
)

fig.update_traces(texttemplate = '%{text:.1f}%', textposition = 'outside')

# a dashed line at 0 marks a day that is the same as a dry day
fig.add_hline(y = 0, line_dash = 'dash', line_color = 'gray')

fig.update_layout(
    title = dict(x = 0.5, xanchor = 'center'),
    height = 500
)

fig.show()

Takeaway: Weather has a small effect on the average fare, and the answer depends on which dry days are used for comparison. Against all dry days, rain raises the average fare slightly, while snow lowers it a little. Against dry days at the same temperature, the snow result flips to an increase, and the rain increases very little.

The only notably large difference here is the significant fare decrease on cold dry dayes vs warmer days (dry days above freezing), which is just a reflection of larger seasonal trends.

### How much do individual days vary?

The averages hide how much individual days differ, so I plot every weather day as a point on top of a box plot, for both comparisons. With so few heavy rain and snow days, it is important to see the spread.

In [96]:
comparison_names = {
    'avg_fare_vs_all': 'All dry days',
    'avg_fare_vs_temp': 'Dry days, same temperature'
}

day_effects = (
    daily[daily['weather_day']]
    [['date', 'condition', 'avg_fare_vs_all', 'avg_fare_vs_temp']]
    .melt(
        id_vars = ['date', 'condition'],
        value_vars = list(comparison_names),
        var_name = 'comparison',
        value_name = 'avg_fare_pct'
    )
)

day_effects['comparison'] = day_effects['comparison'].map(comparison_names)

In [97]:
fig = px.box(
    day_effects,
    x = 'condition',
    y = 'avg_fare_pct',
    color = 'comparison',
    points = 'all',
    category_orders = {'condition': ['Light rain', 'Heavy rain', 'Snow']},
    hover_data = {'date': True},
    title = 'Daily Average Fare Compared to Dry Days',
    labels = {
        'condition': 'Weather Condition',
        'avg_fare_pct': 'Difference in Average Fare (%)',
        'comparison': 'Compared to'
    }
)

# a dashed line at 0 marks a day that is the same as a dry day
fig.add_hline(y = 0, line_dash = 'dash', line_color = 'gray')

fig.update_layout(
    title = dict(x = 0.5, xanchor = 'center'),
    height = 500
)

fig.show()

Takeaway: Again, snow days have the most impact, but it looks like there could be outliers (for example, a handful of expensive holiday days) that are pulling the average up. Against all dry days, snow looks lower because of the seasonal impacts. Overall, the weather effects on the average fare are small compared with the normal day-to-day variation.

### What does cold alone change?

To look at cold without rain or snow, I compare dry days below freezing to dry days above freezing, separately for weekdays and weekends.

Cold days are also winter days, so this comparison mixes the effect of temperature with other seasonal differences. The seasonal section below separates the two.

In [98]:
(
    daily[daily['condition'] == 'Dry']
    .groupby(['day_type', 'below_freezing'])
    .agg(
        days = ('date', 'count'),
        avg_daily_trips = ('trip_count', 'mean'),
        avg_fare = ('avg_fare', 'mean'),
        median_fare = ('median_fare', 'mean'),
        fare_per_mile = ('fare_per_mile', 'mean'),
        avg_miles = ('avg_miles', 'mean'),
        avg_minutes = ('avg_minutes', 'mean')
    )
    .round(2)
)

days  avg_daily_trips  avg_fare  median_fare  \
day_type below_freezing                                                 
Weekday  False            112        139744.67     18.48        13.04   
         True              44        136514.68     16.65        12.44   
Weekend  False             42        175144.67     17.00        12.92   
         True              19        171811.84     15.16        11.71   

                         fare_per_mile  avg_miles  avg_minutes  
day_type below_freezing                                         
Weekday  False                    2.81       6.59        19.67  
         True                     2.79       6.00        17.04  
Weekend  False                    2.93       5.85        17.24  
         True                     2.86       5.34        15.01

Takeaway: Cold dry days have noticeably lower fares. On weekdays the average fare drops from \$18.48 above freezing to \$16.65 below freezing (-9.9%), and on weekends it drops from \$17.00 to \$15.16 (-10.8%). The median fare falls too, but by less (-4.6% on weekdays and -9.4% on weekends). Fare per mile barely changes (\$2.81 to \$2.79 on weekdays and \$2.93 to \$2.86 on weekends), while trips are about 9% shorter in distance and 13% shorter in duration. The lower fares come from shorter trips and not from cheaper prices. Cold days also have slightly fewer trips (about 2% fewer). Cold days are mostly winter days, so some of this difference may be seasonal, which the seasonal section checks.

# Is it pricing or trip mix?

A fare change could come from two places: the price of a trip could change, or the kind of trips people take could change. For example, if people take shorter trips in the snow, the average fare could fall even if prices stayed the same. I look at the fare next to the distance and duration:

| Measure | What a change could mean |
|---|---|
| avg_fare | Higher prices, or a different mix of trips |
| fare_per_mile | A change in price, or a shift toward shorter trips |
| avg_miles | Longer or shorter trips |
| avg_minutes | Slower traffic, or longer or shorter trips |

In the earlier EDA, fare increased with trip distance, so a fare change that comes with a matching distance change is probably a trip-mix effect. Fare per mile is only a rough guide to pricing, because short trips cost more per mile (the base fare is spread over fewer miles).

In [99]:
mix_measures = {
    'avg_fare_pct': 'Average fare',
    'fare_per_mile_pct': 'Fare per mile',
    'avg_miles_pct': 'Average trip miles',
    'avg_minutes_pct': 'Average trip minutes'
}

mix = fare_effects.melt(
    id_vars = ['weather', 'compared_to'],
    value_vars = list(mix_measures),
    var_name = 'measure',
    value_name = 'pct_vs_baseline'
)

mix['measure'] = mix['measure'].map(mix_measures)
mix['scenario'] = mix['weather'] + '<br>vs ' + mix['compared_to']

In [100]:
fig = px.bar(
    mix,
    x = 'scenario',
    y = 'pct_vs_baseline',
    color = 'measure',
    barmode = 'group',
    title = 'Fare, Fare per Mile, Distance, and Duration Compared to Dry Days',
    labels = {
        'scenario': 'Weather and Comparison',
        'pct_vs_baseline': 'Average Difference (%)',
        'measure': 'Measure'
    }
)

fig.add_hline(y = 0, line_dash = 'dash', line_color = 'gray')

fig.update_layout(
    title = dict(x = 0.5, xanchor = 'center'),
    height = 550
)

fig.show()

Takeaway: This reinforces earlier results that seasonal trends are the biggest factors.

## Hourly fare change compared to dry days

This chart shows the percent difference from dry days in the same hour, so it shows when weather matters most.

The heavy rain and snow groups are small, and some hours (like 3 AM) have few trips, so some effects might be exagerated.

In [103]:
hourly_wide = hourly.pivot_table(
    index = ['day_type', 'hour'],
    columns = 'condition',
    values = 'avg_fare',
    observed = True
)

hourly_relative = pd.DataFrame({
    condition: (hourly_wide[condition] / hourly_wide['Dry'] - 1) * 100
    for condition in ['Light rain', 'Heavy rain', 'Snow']
}).reset_index().melt(
    id_vars = ['day_type', 'hour'],
    var_name = 'condition',
    value_name = 'fare_pct_vs_dry'
)

In [104]:
fig = px.line(
    hourly_relative,
    x = 'hour',
    y = 'fare_pct_vs_dry',
    color = 'condition',
    facet_col = 'day_type',
    markers = True,
    color_discrete_map = CONDITION_COLORS,
    category_orders = {'day_type': ['Weekday', 'Weekend']},
    title = 'Hourly Average Fare Compared to Dry Days',
    labels = {
        'hour': 'Hour of Day',
        'fare_pct_vs_dry': 'Difference from Dry Days (%)',
        'condition': 'Weather',
        'day_type': 'Day Type'
    }
)

# a dashed line at 0 marks no difference from dry days
fig.add_hline(y = 0, line_dash = 'dash', line_color = 'gray')

fig.update_xaxes(tickmode = 'array', tickvals = hour_ticks, ticktext = hour_labels)

fig.update_layout(
    title = dict(x = 0.5, xanchor = 'center'),
    height = 500
)

fig.show()

Takeaway: Hourly average fare seems to follow the same shape accross weather type, especially on weekends.

Snow is different on weekdays. Fares are about 10% lower than dry days at midday (10 AM to 3 PM) and about 7.5% lower during the afternoon rush, while the morning rush is up about 4%. Snow days are cold days, so part of the midday drop likely reflects the shorter trips seen on cold days.

The biggest differences are overnight (12 AM to 4 AM). Weekday heavy rain is up 28% to 62% from 12 AM to 2 AM and weekday snow is up 20% to 43%, but those hours have the fewest trips and while the lines jump around I do not think it is necessarily significant.

# How do fares change over the seasons?

## Fares by month

Fares may move over the year even without any weather effect. To see this, I plot the monthly average fare for all days and for dry days only. If the two lines are close together, weather is not what moves the monthly fare.

Monthly averages are trip-weighted: total fares divided by total trips.

In [105]:
monthly = (
    daily
    .groupby('month')
    .agg(
        days = ('date', 'count'),
        weather_days = ('weather_day', 'sum'),
        trips = ('trip_count', 'sum'),
        fare_total = ('fare_total', 'sum'),
        miles_total = ('miles_total', 'sum'),
        seconds_total = ('seconds_total', 'sum')
    )
    .reset_index()
)

monthly['avg_fare'] = monthly['fare_total'] / monthly['trips']
monthly['fare_per_mile'] = monthly['fare_total'] / monthly['miles_total']
monthly['avg_miles'] = monthly['miles_total'] / monthly['trips']
monthly['avg_minutes'] = monthly['seconds_total'] / 60 / monthly['trips']
monthly['month_name'] = monthly['month'].map(month_names)

# average fare using only the dry days in each month
dry_monthly = (
    daily[daily['condition'] == 'Dry']
    .groupby('month')[['fare_total', 'trip_count']]
    .sum()
)

monthly['avg_fare_dry'] = monthly['month'].map(
    dry_monthly['fare_total'] / dry_monthly['trip_count']
)

In [106]:
monthly[[
    'month_name', 'days', 'weather_days', 'trips',
    'avg_fare', 'avg_fare_dry', 'fare_per_mile', 'avg_miles', 'avg_minutes'
]].round(2)

,month_name,days,weather_days,trips,avg_fare,avg_fare_dry,fare_per_mile,avg_miles,avg_minutes
0,Jan,31,14,4402100,14.87,14.29,2.72,5.47,15.53
1,Feb,28,9,4306258,16.13,16.14,2.87,5.62,16.14
2,Mar,31,14,4744154,18.00,17.19,3.06,5.88,17.53
3,Apr,30,13,4522357,17.62,17.81,2.81,6.28,18.65
4,May,31,11,4796310,18.86,18.96,3.08,6.13,19.18
5,Jun,30,14,4515638,19.88,20.28,3.20,6.22,19.28
6,Jul,31,21,4772294,17.12,17.52,2.83,6.05,18.68
7,Aug,31,9,4931679,17.40,16.90,2.87,6.06,18.74
8,Sep,30,8,4349363,17.89,18.20,2.79,6.42,19.15
9,Oct,31,9,4825270,17.55,17.23,2.75,6.39,19.00


In [107]:
monthly_plot = monthly.melt(
    id_vars = ['month', 'month_name'],
    value_vars = ['avg_fare', 'avg_fare_dry'],
    var_name = 'days_included',
    value_name = 'fare'
)

monthly_plot['days_included'] = monthly_plot['days_included'].map({
    'avg_fare': 'All days',
    'avg_fare_dry': 'Dry days only'
})

fig = px.line(
    monthly_plot,
    x = 'month',
    y = 'fare',
    color = 'days_included',
    markers = True,
    title = 'Average Rideshare Fare by Month in Chicago, 2025',
    labels = {
        'month': 'Month',
        'fare': 'Average Fare ($)',
        'days_included': 'Days included'
    }
)

fig.update_xaxes(
    tickmode = 'array',
    tickvals = list(month_names),
    ticktext = list(month_names.values())
)

fig.update_layout(
    title = dict(x = 0.5, xanchor = 'center'),
    height = 500
)

fig.show()

Takeaway: Fares change a lot over the year, but the daily weather doesn't seem to have a big effect. The monthly average fare is lowest in January (\$14.87) and highest in June (\$19.88), but this is probably based more on overall travel patterns (tourism or summer events) than faily weather patterns. The all-days line and the dry-days-only line are very close, so removing the rain and snow days barely changes the monthly average.

